In [ ]:
from langgraph.graph import START ,END,StateGraph
from langgraph.types import Send
from pydantic import BaseModel , Field
from typing import TypedDict,Annotated,Optional,Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage,SystemMessage,BaseMessage
import operator
from langchain_community.tools.tavily_search import TavilySearchResults
from pathlib import Path
import uuid
load_dotenv()
import os 

In [ ]:
os.environ['LANGCHAIN_PROJECT']="blog-writer"

In [ ]:
model = ChatGroq(model="openai/gpt-oss-20b",
    temperature=0.7)

In [ ]:
class Task(BaseModel):
    Task_id : str
    task_title:str
    Task_text:str = Field(description="Text for Task in md according to task Title")
    bullets:list[str]=Field(min_length=3,max_length=5,description="3-5 lines about the things expected to be covered in this task")
    target_word:int=Field(description="Target word count for this section between 150-400")
    section_types:Literal["intro","core","examples","common-mistakes","outro","checklist","conclusion"]=Field(description="common-mistake has to be present only a single time")
    requires_research:bool=Field(description="research needed for this task")
    requires_citations:bool=Field(description="Citiation needed for this task")
    requires_code:bool=Field(description="code needed for this task?")

In [ ]:
class Router_schema(BaseModel):
    OuterKnowledge:bool=Field(description="need of outer knowledge,like browser")
    knowledge_type:Literal['No dependency',"Hybrid","Full dependency"]
    queries:list[str]=Field(default_factory=list,description="3 to 5 research queries when external research is needed; otherwise an empty list")

In [ ]:
class Blog_Plan(BaseModel):
    Blog_title:str = Field(description="Title of the Blog")
    tone:str = Field(description="tone of the Blog for example-Technical,Motivative etc.")
    target_audience:str=Field(description="Who is this blog for:")
    Blog_Sections:list[Task] = Field(description="List of Blog Sections")

In [ ]:
class evidence_item(BaseModel):
    title: str
    url: str
    research_content:  Optional[str]
    published_at :  Optional[str]
    source:Optional[str]  


In [ ]:
class Research_Evidence_Pack(BaseModel):
    evidence_items:list[evidence_item]

In [ ]:
class graph_State(TypedDict,total=False):
    Title : str
    OuterKnowledge:Optional[bool]
    Evidences:Optional[list[evidence_item]]
    knowledge_type:Optional[str]
    queries:Optional[list[str]]
    Plan:Optional[Blog_Plan]
    temp_Blog:Annotated[list[str],operator.add]
    final_Blog:Optional[str]

In [ ]:
def research(state:graph_State):
    queries=state['queries']
    
    tavily_tool = TavilySearchResults(max_results=1)
    prompt = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of evidence items.

Return only valid JSON matching this shape:
{"evidence_items":[{"title":"string","url":"string","research_content":"string or null","published_at":"YYYY-MM-DD or null","source":"string or null"}]}

Rules:
- Only include items with a non-empty url.
- Prefer relevant and authoritative sources.
- If a published date is missing or unclear, use null. Do not guess.
- Keep research_content short, no more than 100 words.
- Deduplicate by URL.
- Use standard JSON escaping.
"""
    raw_search_Results:list[dict]=[]
    for i in queries:
        raw_search_Results.append({"evidences":tavily_tool.invoke(i)})

    research_model = ChatGroq(
        model="qwen/qwen3.8-27b",
        temperature=0
    )
    llm = research_model.with_structured_output(Research_Evidence_Pack, method="json_mode")
    res = llm.invoke(
        [SystemMessage(content=prompt),HumanMessage(content=f"Here are the raw web search results: {raw_search_Results}")]
    )

    dedup = {}
    for i in res.evidence_items:
        if i.url:
            dedup[i.url]=i

    return {"Evidences":list(dedup.values())}

In [ ]:
def router(state:graph_State):
    Title=state['Title']
    system_prompt = """
                          You are a research-requirement decision maker for a blog-writing workflow.

                          Determine whether the blog titled below can be written accurately using the
                          language model's existing knowledge or whether external research is required.

                          Use these rules:
                          - Set OuterKnowledge to false when the topic is stable, general, and well-known.
                          - Set OuterKnowledge to true when the topic requires current, niche, factual,
                            statistical, regulatory, scientific, product-specific, or rapidly changing
                            information.
                          - Choose "No dependency" when external research is unnecessary.
                          - Choose "Hybrid" when most of the blog can use existing knowledge but some
                            claims require verification.
                          - Choose "Full dependency" when reliable external sources are essential for
                            most of the blog.
                          - When OuterKnowledge is false, return an empty queries list.
                          - When OuterKnowledge is true, generate 3 to 5 specific research queries.
                          - Research queries must directly support the blog title and should seek
                            authoritative and up-to-date sources.
                          - Do not perform the research. Only decide whether research is needed.
                          - Return only a response matching the Router_schema.
"""
    result = model.with_structured_output(Router_schema).invoke([
              SystemMessage(content=system_prompt),
              HumanMessage(content=f"Blog title: {Title}")
          ])
    return {
          "OuterKnowledge":result.OuterKnowledge,
      "knowledge_type":result.knowledge_type,
      "queries":result.queries
      }

def router_need(state:graph_State):
    return "research" if state['OuterKnowledge'] else "planner"

In [ ]:
def planner(state: graph_State):
    
    title = state["Title"]
    evidences = state.get('Evidences') or None
    system_prompt = f"""
You are an expert blog planner.

Create a detailed and logically ordered plan for a blog post based on the
title provided by the user.

Requirements:
- Generate between 3 and 4 sections.
- Return only content that matches the Blog_Plan schema.
- Give every section a clear, unique title.
- Provide 3 to 5 concise bullet points for every section.
- Include an appropriate target word count between 150 and 400 for each section.
- Use each section type appropriately.
- Use the "common-mistakes" section type at most once.
- Ensure the sections have a natural progression: introduction, core concepts,
  examples or applications, common mistakes where useful, and conclusion.
- Avoid duplicate sections and unnecessary information.

also use this information as the evidence's which came from the internet , which can help you write a factually and upto date blog-{evidences}
"""

    result = model.with_structured_output(Blog_Plan).invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Create a blog plan for this title: {title}")
    ])

    return {"Plan": result}

In [ ]:
def fanout(state:graph_State):
        return [
        Send(
            "worker",
            {
                "task": work,
                "topic": state["Title"],
                "plan": state['Plan'],
            },
        )
        for work in state["Plan"].Blog_Sections
    ]

In [ ]:
def worker(payload: dict):
    task = payload["task"]
    plan = payload["plan"]
    worker_model = ChatGroq(
        model="qwen/qwen3.8-27b",
        temperature=0.7
    )
    work_res = worker_model.invoke([
        SystemMessage(
            content="""You are an expert blog writer.

Generate one complete blog section based on the topic and instructions provided by the user.

Follow these rules:
- Return only the final Markdown content.
- Start with a suitable Markdown heading for the section.
- Write only the requested section, not the entire blog.
- Follow the specified section type, title, task brief, and required bullet points.
- Stay close to the requested target word count.
- Use clear, accurate, and well-structured explanations.
- Match the blog’s tone and target audience.
- Include examples, lists, tables, or code blocks only when they improve clarity.
- Avoid repeating information from other sections.
- Do not mention these instructions, the planning process, or being an AI.
- Do not add introductory or concluding commentary outside the requested section."""
        ),
        HumanMessage(
            content=(
                f"Write Markdown for the blog title: {plan.Blog_title}\n"
                f"Topic: {task.task_title}\n"
                f"Brief: {task.Task_text}\n" 
                f"bullets:{task.bullets}\n" 
                f"target_words:{task.target_word}\n"
                f"section_type:{task.section_types}\n" 
            )
        ),
    ])

    return {"temp_Blog": [work_res.content]}

In [ ]:
def writer(state:graph_State)->dict:
    Blog_title=state['Title']
    Blog_Sections = "\n\n".join(state['temp_Blog']).strip()

    md_file = f"#{Blog_title}\n\n{Blog_Sections}"

    filename = Blog_title.lower().replace("", "_")+".md"
    output_path = Path(filename)
    output_path.write_text(md_file,encoding="utf-8")

    return ({"final_Blog":md_file})

In [ ]:
Graph = StateGraph(graph_State)
Graph.add_node("router",router)
Graph.add_node("research",research)
Graph.add_node("planner",planner)
Graph.add_node("worker",worker)
Graph.add_node("writer",writer)

Graph.add_edge(START,"router")
Graph.add_conditional_edges("router",router_need,{
    "research":"research",
    "planner":"planner"
})
Graph.add_edge("research","planner")
Graph.add_edge("router","planner")
Graph.add_conditional_edges("planner",fanout,['worker'])
Graph.add_edge("worker","writer")

Main_graph = Graph.compile()



In [ ]:
Main_graph

In [ ]:
res = Main_graph.invoke({
    "Title": "rise of open ai models from 2022 to 2026"  
})

res['final_Blog']

In [ ]:
res['Plan']

In [ ]:
res['queries']

In [ ]:
res

In [ ]:
res['temp_Blog']